Topic E: Movie Review Sentiment Analysis

Goal: Determine whether user reviews express positive or negative sentiment.

Dataset: IMDB Movie Reviews Dataset.

Tasks: Clean review text, extract n-gram features, fit a classification model, and run error analysis on misclassified reviews.


#Importing

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [9]:
data=pd.read_csv('/content/IMDB Dataset.csv')

In [10]:
data.head(10)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [11]:
data.info()
#no null values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [12]:
data['review']=data['review'].str.replace('<br /><br />',"",regex=False)

In [13]:
data['review'][3]

"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them."

In [14]:
data["sentiment"] = data["sentiment"].map({
    "positive": 1,
    "negative": 0
})

#Cleaning

In [15]:

import re

def clean_text(x):
    x = x.lower()

    # Preserve ratings
    x = re.sub(r"(\d+)\s*/\s*(\d+)", r"rating_\1_out_of_\2", x)

    # Remove HTML
    x = re.sub(r"<.*?>", " ", x)

    # Keep letters, numbers and underscores
    x = re.sub(r"[^a-z0-9_\s]", " ", x)

    # Remove extra spaces
    x = re.sub(r"\s+", " ", x)

    return x.strip()

data["clean_review"] = data["review"].apply(clean_text)

from sklearn.feature_extraction.text import TfidfVectorizer

#Term Frequency × Inverse Document Frequency
#common everywhere → lower importance
#important/rarer term → higher importance

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2
)

X = tfidf.fit_transform(data["clean_review"])

An n-gram is a sequence of n words.

#Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Split RAW reviews
X_train, X_test, y_train, y_test = train_test_split(
    data["review"],
    data["sentiment"],
    test_size=0.2,
    random_state=42
)

# 2. Create TF-IDF
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2
)

# 3. Fit ONLY on training data
X_train_vec = tfidf.fit_transform(X_train)

# 4. Transform test data
X_test_vec = tfidf.transform(X_test)

# 5. Train model
model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)


In [ ]:
y_pred = model.predict(X_test_vec)

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score,accuracy_score,f1_score

cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("Precision:", precision)
print("Recall:", recall)


#thats a really good metric

In [ ]:
from sklearn.svm import LinearSVC

svc = LinearSVC()
svc.fit(X_train_vec, y_train)
svc_pred = svc.predict(X_test_vec)


print("Accuracy:", accuracy_score(y_test, svc_pred))
print("Precision:", precision_score(y_test, svc_pred))
print("Recall:", recall_score(y_test, svc_pred))
print("F1 Score:", f1_score(y_test, svc_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_vec, y_train)

rf_pred = rf.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1 Score:", f1_score(y_test, rf_pred))

Error Analysis

In [ ]:
import pandas as pd

error_df = pd.DataFrame({
    "review": X_test,
    "actual": y_test,
    "predicted": svc_pred
})

error_df = error_df[
    error_df["actual"] != error_df["predicted"]
]
#contains all the error nodes
print(error_df.head(10))

In [ ]:
false_positive = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
]
#proportion of all actual negatives that are incorrectly flagged as positive.
print(false_positive.head())

In [ ]:
false_negative = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
]
#proportion of all actual positives that are incorrectly flagged as negative.
print(false_negative.head())

In [ ]:
print("False Positives:", len(false_positive))
print("False Negatives:", len(false_negative))

In [ ]:
import joblib
joblib.dump(svc,"model.pkl")
joblib.dump(tfidf, "tfidf.pkl")